In [119]:
%load_ext autoreload
%autoreload 2


import matplotlib.pyplot as plt
import numpy as np

from darkmod import laue
from darkmod.beam import GaussianLineBeam
from darkmod.crl import CompundRefractiveLens
from darkmod.crystal import Crystal
from darkmod.detector import Detector
from darkmod.resolution import PentaGauss

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [120]:
plt.style.use("dark_background")
fontsize = 16  # General font size for all text
ticksize = 16  # tick size
plt.rcParams["font.size"] = fontsize
plt.rcParams["xtick.labelsize"] = ticksize
plt.rcParams["ytick.labelsize"] = ticksize

In [121]:
defgrad = np.load("defgrad.npy").transpose((2, 3, 4, 0, 1))
voxel_size = 0.063  # microns

In [122]:
number_of_lenses = 88
lens_space = 1600  # microns
lens_radius = 50  # microns
magnification = 11.68
energy = 19.2  # keV
Z = 4  # atomic number, berillium
rho = 1.845  # density, berillium, g/cm^3
A = 9.0121831  # atomic mass number, berillium, g/mol

# hkl = np.array([0, 0, 2])
hkl = np.array([-2, 2, 0])

eta = np.radians(0)

unit_cell = [3.52, 3.52, 3.52, 90.0, 90.0, 90.0]  # angstrom
H = np.array([[1, -1, 0], [1, 1, 0], [0, 0, 1]]).T
orientation = H / np.linalg.norm(H, axis=0)

z_std = 0.3  # microns
crystal = Crystal(unit_cell, orientation)
d = np.pi * 2 / np.linalg.norm(crystal.U @ crystal.B @ hkl)
theta = np.arcsin(laue.keV_to_angstrom(energy) / (2 * d))


# Beam divergence params
beam_FWHM_vertical = 0.027 * 1e-3
beam_FWHM_horizontal = 1e-9

# Beam wavelength broadening
sigma_e = (6 * 1e-5) / (2 * np.sqrt(2 * np.log(2)))

# crl acceptance
FWHM_CRL_vertical = 0.556 * 1e-3
FWHM_CRL_horizontal = FWHM_CRL_vertical

# Detector size
det_row_count = 300
det_col_count = 250
pixel_size = 0.65 / 2.0  # microns, effective pixel size with optics.
super_sampling = 2
dynamic_range = 2**16 - 1
exposure = 500
psf_width = 1
noise = True

spatial_artefact = False

phi_0 = -theta
phi_step = np.radians(2 * 1e-3) / 8
delta_phi_values = np.arange(0, 80 * phi_step, phi_step)
delta_phi_values -= np.median(delta_phi_values)
phi_values = phi_0 + delta_phi_values

In [123]:
dx, dy, dz = np.array(defgrad.shape[0:3]) * voxel_size
xg = np.arange(0, dx, voxel_size)
yg = np.arange(0, dy, voxel_size)
zg = np.arange(0, dz, voxel_size)
xg -= np.median(xg)
yg -= np.median(yg)
zg -= np.median(zg)
Xgrid, Ygrid, Zgrid = np.meshgrid(xg, yg, zg, indexing="ij")
crystal.discretize(Xgrid, Ygrid, Zgrid, defgrad)


In [124]:
crystal.align(hkl, axis=np.array([0, 0, 1]))
crystal.align(np.array([-1, -1, 0]), axis=np.array([1, 0, 0]), transformation_hkl=hkl)
crystal.goniometer.relative_move(dphi=0, dchi=0, domega=0, dmu=phi_0)

In [125]:
# np.linalg.inv(crystal.U @ crystal.B) @ np.array([0,0,1])
delta = laue.refractive_decrement(Z, rho, A, energy)
crl = CompundRefractiveLens(
    number_of_lenses, lens_space, lens_radius, delta, magnification
)
crl.goto(theta, eta)
lambda_0 = laue.keV_to_angstrom(energy)
beam = GaussianLineBeam(z_std=z_std, energy=energy)  # 100 nm = 0.1 microns

In [126]:
epsilon = np.random.normal(0, sigma_e, size=(20000,))
random_energy = energy + epsilon * energy
sigma_lambda = laue.keV_to_angstrom(random_energy).std()

resolution_function = PentaGauss(
    crl.optical_axis,
    beam_FWHM_horizontal / (2 * np.sqrt(2 * np.log(2))),
    # desired_FWHM_N / (2 * np.sqrt(2 * np.log(2))),
    beam_FWHM_vertical / (2 * np.sqrt(2 * np.log(2))),
    FWHM_CRL_horizontal / (2 * np.sqrt(2 * np.log(2))),
    FWHM_CRL_vertical / (2 * np.sqrt(2 * np.log(2))),
    lambda_0,
    sigma_lambda,
)
resolution_function.compile()


In [127]:
detector = Detector.wall_mount(
    crl,
    pixel_size,
    det_row_count,
    det_col_count,
    super_sampling=super_sampling,
    exposure=exposure,
    dynamic_range=dynamic_range,
    psf_width=psf_width,
    noise=noise,
)

In [ ]:
mu_steps = np.radians(np.linspace(-0.08, 0.08, 20))  # rocking
chi_steps = np.radians(np.linspace(-0.08, 0.08, 20))  # rolling
mu_steps -= np.median(mu_steps)
chi_steps -= np.median(chi_steps)
mu_steps, chi_steps

(array([-1.39626340e-03, -1.24928831e-03, -1.10231321e-03, -9.55338117e-04,
        -8.08363022e-04, -6.61387927e-04, -5.14412832e-04, -3.67437737e-04,
        -2.20462642e-04, -7.34875475e-05,  7.34875475e-05,  2.20462642e-04,
         3.67437737e-04,  5.14412832e-04,  6.61387927e-04,  8.08363022e-04,
         9.55338117e-04,  1.10231321e-03,  1.24928831e-03,  1.39626340e-03]),
 array([-1.39626340e-03, -1.24928831e-03, -1.10231321e-03, -9.55338117e-04,
        -8.08363022e-04, -6.61387927e-04, -5.14412832e-04, -3.67437737e-04,
        -2.20462642e-04, -7.34875475e-05,  7.34875475e-05,  2.20462642e-04,
         3.67437737e-04,  5.14412832e-04,  6.61387927e-04,  8.08363022e-04,
         9.55338117e-04,  1.10231321e-03,  1.24928831e-03,  1.39626340e-03]))

In [152]:
mu0 = crystal.goniometer.mu
chi0 = crystal.goniometer.chi
omega0 = crystal.goniometer.omega
phi0 = crystal.goniometer.phi

mosa = np.zeros(
    (detector.det_col_count, detector.det_row_count, len(mu_steps), len(chi_steps))
)

for i, dmu in enumerate(mu_steps):
    for j, dchi in enumerate(chi_steps):
        crystal.goniometer.goto(
            phi=phi0,
            chi=chi0 + dchi,
            omega=omega0,
            mu=mu0 + dmu,
        )

        mosa[..., i, j] = crystal.diffract(
            hkl,
            resolution_function,
            crl,
            detector,
            beam,
            spatial_artefact=spatial_artefact,
        )

crystal.goniometer.goto(
    phi=phi0,
    chi=chi0,
    omega=omega0,
    mu=mu0,
)


In [154]:
from ipywidgets import IntSlider, interact

n, m, d, _ = mosa.shape


def show_slice(idx):
    plt.figure(figsize=(12, 14))
    plt.imshow(mosa[:, :, idx, mosa.shape[3] // 2], cmap="viridis")
    plt.title(f"Slice {idx}")
    plt.colorbar(fraction=0.03, pad=0.04)
    plt.show()


interact(show_slice, idx=IntSlider(min=0, max=d - 1, step=1, value=0))

interactive(children=(IntSlider(value=0, description='idx', max=19), Output()), _dom_classes=('widget-interact…

<function __main__.show_slice(idx)>